In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [85]:
PACK_NUMBER = 10 # ← Change this to 1, 2, 3 ... 10 each session
BATCH_SIZE = 64 # do not change
HASH_SIZE = 8   # Produces 64-bit hashes — do not change
NUM_WORKERS = 4  # Threads for parallel hashing — do not change



KAGGLE_OUTPUT = '/kaggle/working'
IMAGES_PER_PACK = 100_000

# Each pack has its own folder: images0 = Pack 1, images1 = Pack 2, etc.
KAGGLE_INPUT = f'/kaggle/input/datasets/sohangundoju/mirflickr-1m/images{PACK_NUMBER - 1}/images'


In [16]:
!pip install imagehash Pillow tqdm --quiet


In [3]:
import os
import glob
import pickle
import concurrent.futures

import numpy as np
import imagehash
from PIL import Image, ImageEnhance
from tqdm import tqdm

print(" All imports successful")


 All imports successful


In [86]:
# CELL 4 — SET UP OUTPUT PATHS

HASH_OUT_DIR = os.path.join(KAGGLE_OUTPUT, 'features', 'hashes')
os.makedirs(HASH_OUT_DIR, exist_ok=True)

output_pkl = os.path.join(HASH_OUT_DIR, f'pack_{PACK_NUMBER}_hashes.pkl')

print(f" Output path: {HASH_OUT_DIR}")

 Output path: /kaggle/working/features/hashes


In [87]:
# CELL 5 — CHECK IF THIS PACK IS ALREADY DONE

if os.path.exists(output_pkl):
    print(f" Pack {PACK_NUMBER} already hashed:")
    print(f"  {output_pkl}")
    print("  Change PACK_NUMBER in Cell 1 to process next pack.")
    raise SystemExit("Already processed.")

print(f"Pack {PACK_NUMBER} not yet processed. Proceeding.")

Pack 10 not yet processed. Proceeding.


In [88]:
# CELL 6 — FIND IMAGES FOR THIS PACK
# No slicing needed — each pack folder contains exactly that pack's images.

print(f"Scanning Pack {PACK_NUMBER} at {KAGGLE_INPUT} ...")

all_patterns = ['**/*.jpg', '**/*.jpeg', '**/*.png']
all_image_paths = []
for pattern in all_patterns:
    all_image_paths.extend(
        glob.glob(os.path.join(KAGGLE_INPUT, pattern), recursive=True)
    )

PACK_PATHS = sorted(list(set([os.path.abspath(p) for p in all_image_paths])))
pack_start = (PACK_NUMBER - 1) * IMAGES_PER_PACK
pack_end = pack_start + len(PACK_PATHS)

print(f" Pack {PACK_NUMBER}:")
print(f" Images : {len(PACK_PATHS)}")
print(f" ID range: {pack_start} → {pack_end - 1}")
print(f" First : {os.path.basename(PACK_PATHS[0])}")
print(f" Last : {os.path.basename(PACK_PATHS[-1])}")

if len(PACK_PATHS) == 0:
    raise ValueError(
        f"No images found at {KAGGLE_INPUT}. "
        "Check that the dataset is attached and PACK_NUMBER is correct."
    )

if len(PACK_PATHS) < 90_000:
    print(f" WARNING: Only {len(PACK_PATHS)} images found — expected ~100,000.")

Scanning Pack 10 at /kaggle/input/datasets/sohangundoju/mirflickr-1m/images9/images ...
 Pack 10:
 Images : 100000
 ID range: 900000 → 999999
 First : 900000.jpg
 Last : 999999.jpg


In [89]:
# CELL 7 — SYNC CHECK
# Share this output in the group chat.

print("=" * 60)
print(f"  SYNC CHECK — Pack {PACK_NUMBER} — share this in group chat")
print("=" * 60)
print(f"  Pack images      : {len(PACK_PATHS)}")
print(f"  ID range         : {pack_start} → {pack_end - 1}")
print(f"  First file       : {os.path.basename(PACK_PATHS[0])}")
print(f"  Last  file       : {os.path.basename(PACK_PATHS[-1])}")
if len(PACK_PATHS) > 500:
    print(f"  File #500        : {os.path.basename(PACK_PATHS[500])}")
if len(PACK_PATHS) > 50000:
    print(f"  File #50000      : {os.path.basename(PACK_PATHS[50000])}")
print("=" * 60)
print("  If any filename differs — stop and debug before extracting.")
print("=" * 60)

  SYNC CHECK — Pack 10 — share this in group chat
  Pack images      : 100000
  ID range         : 900000 → 999999
  First file       : 900000.jpg
  Last  file       : 999999.jpg
  File #500        : 900500.jpg
  File #50000      : 950000.jpg
  If any filename differs — stop and debug before extracting.


In [90]:
# CELL 8 — SHARED IMAGE LOADING UTILITY
# DO NOT MODIFY ANYTHING IN THIS CELL.

IMAGENET_MEAN = np.array([0.485, 0.456, 0.406], dtype=np.float32)
IMAGENET_STD  = np.array([0.229, 0.224, 0.225], dtype=np.float32)


def load_single_image(path: str) -> np.ndarray:
    """
    Loads and preprocesses one image for ResNet-50.
    Returns float32 array of shape (3, 224, 224).
    Raises exception on failure — handled by caller.
    """
    img = Image.open(path).convert('RGB')
    img = img.resize((224, 224), Image.BILINEAR)
    arr = np.array(img, dtype=np.float32)
    arr /= 255.0
    arr -= IMAGENET_MEAN
    arr /= IMAGENET_STD
    arr  = arr.transpose(2, 0, 1)   # (H,W,C) → (C,H,W)
    return arr


def load_paths_in_batches(image_paths: list, id_offset: int, batch_size: int):
    """
    Generator — yields (global_ids, image_arrays) until all paths exhausted.
    Failed images → zero array so all files stay the same length.
    """
    failed_log = f"/kaggle/working/failed_pack_{PACK_NUMBER}.txt"
    total      = len(image_paths)

    print(f"[loader] {total} images, IDs {id_offset} → {id_offset + total - 1}")

    for batch_start in range(0, total, batch_size):
        batch_paths       = image_paths[batch_start : batch_start + batch_size]
        actual_batch_size = len(batch_paths)
        global_ids        = list(range(id_offset + batch_start,
                                       id_offset + batch_start + actual_batch_size))
        image_arrays      = np.zeros((actual_batch_size, 3, 224, 224), dtype=np.float32)

        for i, path in enumerate(batch_paths):
            try:
                image_arrays[i] = load_single_image(path)
            except Exception as e:
                with open(failed_log, 'a') as f:
                    f.write(f"{path}\t{e}\n")

        yield global_ids, image_arrays

print(" Shared loading utility defined.")


 Shared loading utility defined.


In [91]:
# CELL 9 — DEFINE HASH FUNCTIONS

def compute_all_hashes(args):
    """
    Computes aHash, dHash, pHash for one image.
    Takes (image_path, global_id) tuple — required for ThreadPoolExecutor.
    Returns (global_id, {ahash, dhash, phash}) or zeros on failure.
    """
    image_path, global_id = args
    failed_log = f"/kaggle/working/failed_pack_{PACK_NUMBER}.txt"

    try:
        img = Image.open(image_path).convert('RGB')

        ah = imagehash.average_hash(img, hash_size=HASH_SIZE)
        dh = imagehash.dhash(img,        hash_size=HASH_SIZE)
        ph = imagehash.phash(img,        hash_size=HASH_SIZE)

        # str() gives hex string → int(..., 16) converts to integer
        return (global_id, {
            'ahash': int(str(ah), 16),
            'dhash': int(str(dh), 16),
            'phash': int(str(ph), 16)
        })

    except Exception as e:
        with open(failed_log, 'a') as f:
            f.write(f"{image_path}\t{e}\n")
        return (global_id, {'ahash': 0, 'dhash': 0, 'phash': 0})


def hamming_distance(h1: int, h2: int) -> int:
    """Number of bit positions that differ between two hash integers."""
    return bin(h1 ^ h2).count('1')


def hash_similarity(h1: int, h2: int) -> float:
    """Similarity 0–1. 1 = identical, 0 = completely different."""
    return 1.0 - (hamming_distance(h1, h2) / 64.0)


print(" Hash functions defined.")



 Hash functions defined.


In [92]:
# CELL 10 — BUILD (PATH, ID) LIST

args_list = [(path, pack_start + i) for i, path in enumerate(PACK_PATHS)]

print(f" Built args list: {len(args_list)} images")
print(f"  First: ID={args_list[0][1]}, file={os.path.basename(args_list[0][0])}")
print(f"  Last : ID={args_list[-1][1]}, file={os.path.basename(args_list[-1][0])}")


 Built args list: 100000 images
  First: ID=900000, file=900000.jpg
  Last : ID=999999, file=999999.jpg


In [93]:
# CELL 11 — RUN PARALLEL HASHING
# Uses ThreadPoolExecutor — hashing is I/O bound (reading image files)
# so threads work well. PIL releases the GIL allowing true concurrency.

hash_dict = {}

with concurrent.futures.ThreadPoolExecutor(max_workers=NUM_WORKERS) as executor:
    results = list(tqdm(
        executor.map(compute_all_hashes, args_list),
        total=len(args_list),
        desc=f"Hashing Pack {PACK_NUMBER}",
        unit="img"
    ))

for global_id, hashes in results:
    hash_dict[global_id] = hashes

print(f"\n Hashing complete.")
print(f"  Total entries : {len(hash_dict)}")
print(f"  ID range      : {min(hash_dict.keys())} → {max(hash_dict.keys())}")

failures = sum(1 for v in hash_dict.values() if v['ahash'] == 0 and v['phash'] == 0)
print(f"  Failed images : {failures}")

Hashing Pack 10: 100%|██████████| 100000/100000 [07:25<00:00, 224.31img/s]



 Hashing complete.
  Total entries : 100000
  ID range      : 900000 → 999999
  Failed images : 0


In [94]:
# CELL 12 — VERIFY HASH QUALITY

import random

print("Hash quality check — Hamming distances for brightness-adjusted pairs:")
print("(Should all be under 10)\n")

sample_paths = random.sample(PACK_PATHS, 5)

for path in sample_paths:
    try:
        img     = Image.open(path).convert('RGB')
        img_mod = ImageEnhance.Brightness(img).enhance(1.1)

        orig_ah = int(str(imagehash.average_hash(img,     hash_size=HASH_SIZE)), 16)
        mod_ah  = int(str(imagehash.average_hash(img_mod, hash_size=HASH_SIZE)), 16)

        orig_dh = int(str(imagehash.dhash(img,     hash_size=HASH_SIZE)), 16)
        mod_dh  = int(str(imagehash.dhash(img_mod, hash_size=HASH_SIZE)), 16)

        orig_ph = int(str(imagehash.phash(img,     hash_size=HASH_SIZE)), 16)
        mod_ph  = int(str(imagehash.phash(img_mod, hash_size=HASH_SIZE)), 16)

        da = hamming_distance(orig_ah, mod_ah)
        dd = hamming_distance(orig_dh, mod_dh)
        dp = hamming_distance(orig_ph, mod_ph)

        status = "Correct" if max(da, dd, dp) < 10 else " WARNING"
        print(f"  {status}  {os.path.basename(path)}")
        print(f"         aHash dist: {da}/64  |  dHash dist: {dd}/64  |  pHash dist: {dp}/64")

    except Exception as e:
        print(f"  Could not test {os.path.basename(path)}: {e}")

Hash quality check — Hamming distances for brightness-adjusted pairs:
(Should all be under 10)

  Correct  987193.jpg
         aHash dist: 0/64  |  dHash dist: 0/64  |  pHash dist: 0/64
  Correct  931203.jpg
         aHash dist: 0/64  |  dHash dist: 0/64  |  pHash dist: 2/64
  Correct  980992.jpg
         aHash dist: 1/64  |  dHash dist: 1/64  |  pHash dist: 2/64
  Correct  976144.jpg
         aHash dist: 0/64  |  dHash dist: 1/64  |  pHash dist: 0/64
  Correct  985615.jpg
         aHash dist: 0/64  |  dHash dist: 2/64  |  pHash dist: 2/64


In [95]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 13 — SAVE OUTPUT
# ─────────────────────────────────────────────────────────────────────────────

with open(output_pkl, 'wb') as f:
    pickle.dump(hash_dict, f)

print(f" Saved: {output_pkl}")
print(f"  Size    : {os.path.getsize(output_pkl) / 1e6:.1f} MB")
print(f"  Entries : {len(hash_dict)}")
print()
print("=" * 60)
print(f"  Pack {PACK_NUMBER} COMPLETE")
print(f"  File saved to /kaggle/working/features/hashes/")
print()
print(f"  ► Download from Output panel (right sidebar):")
print(f"    pack_{PACK_NUMBER}_hashes.pkl")
print()
print(f"  ► Change PACK_NUMBER to {PACK_NUMBER + 1} for next session")
print("=" * 60)

 Saved: /kaggle/working/features/hashes/pack_10_hashes.pkl
  Size    : 4.7 MB
  Entries : 100000

  Pack 10 COMPLETE
  File saved to /kaggle/working/features/hashes/

  ► Download from Output panel (right sidebar):
    pack_10_hashes.pkl

  ► Change PACK_NUMBER to 11 for next session


In [99]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 14 — MERGE (RUN ONCE AFTER ALL 10 PACKS ARE DONE)
#
# Before running:
# - Upload all 10 pack_N_hashes.pkl files as a Kaggle dataset
#   named 'mirflickr-hash-packs'
# - Attach via Settings → Add data → Your datasets
# - Files will be at /kaggle/input/mirflickr-hash-packs/
# ─────────────────────────────────────────────────────────────────────────────

PACKS_INPUT_DIR = '/kaggle/input/datasets/wari30/mirflickr-hash-packs'

print("Checking all 10 pack files exist before merging ...")
missing = []
for n in range(1, 11):
    path = os.path.join(PACKS_INPUT_DIR, f'pack_{n}_hashes.pkl')
    if not os.path.exists(path):
        missing.append(n)

if missing:
    raise FileNotFoundError(
        f"Missing pack files for packs: {missing}.\n"
        "Upload all 10 pack_N_hashes.pkl files as a Kaggle dataset first."
    )

print(" All 10 pack files found. Starting merge ...")

merged_hashes = {}

for n in range(1, 11):
    pkl_path = os.path.join(PACKS_INPUT_DIR, f'pack_{n}_hashes.pkl')
    with open(pkl_path, 'rb') as f:
        pack_hashes = pickle.load(f)
    merged_hashes.update(pack_hashes)
    print(f"  Pack {n} loaded — {len(pack_hashes)} entries.")

print(f"\n Merge complete. Total entries: {len(merged_hashes)}")

merged_path = '/kaggle/working/all_perceptual_hashes.pkl'
with open(merged_path, 'wb') as f:
    pickle.dump(merged_hashes, f)

print(f" Saved: {merged_path}")
print(f"  Size : {os.path.getsize(merged_path) / 1e6:.1f} MB")


Checking all 10 pack files exist before merging ...
 All 10 pack files found. Starting merge ...
  Pack 1 loaded — 100000 entries.
  Pack 2 loaded — 100000 entries.
  Pack 3 loaded — 100000 entries.
  Pack 4 loaded — 100000 entries.
  Pack 5 loaded — 100000 entries.
  Pack 6 loaded — 100000 entries.
  Pack 7 loaded — 100000 entries.
  Pack 8 loaded — 100000 entries.
  Pack 9 loaded — 100000 entries.
  Pack 10 loaded — 100000 entries.

 Merge complete. Total entries: 1000000
 Saved: /kaggle/working/all_perceptual_hashes.pkl
  Size : 54.8 MB
